In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import SymbolDataset, SymbolClassifier

In [8]:
# Example placeholders
sf = 9
input = 64
hidden = 1024
output = 512

folder_path = "classifier_dataset_sf{}_{}_{}".format(sf, input, output)

X = np.load(f"{folder_path}/X.npy")   # shape (30720, 16)
y = np.load(f"{folder_path}/y.npy")   # shape (30720,)

dataset = SymbolDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

layers = [input,hidden,output]
model = SymbolClassifier(layers).to(device)
criterion = nn.CrossEntropyLoss()   # Softmax included
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [9]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)              # (batch, 512)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    acc = correct / total * 100

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")
    
torch.save(model.state_dict(), "symbol_classifier.pt")

Epoch [1/10] Loss: 3.6370 | Accuracy: 10.14%
Epoch [2/10] Loss: 2.7549 | Accuracy: 17.02%
Epoch [3/10] Loss: 2.4771 | Accuracy: 22.31%
Epoch [4/10] Loss: 2.3012 | Accuracy: 26.04%
Epoch [5/10] Loss: 2.1760 | Accuracy: 29.30%
Epoch [6/10] Loss: 2.0748 | Accuracy: 31.63%
Epoch [7/10] Loss: 1.9912 | Accuracy: 33.96%
Epoch [8/10] Loss: 1.9193 | Accuracy: 36.15%
Epoch [9/10] Loss: 1.8605 | Accuracy: 37.79%
Epoch [10/10] Loss: 1.8037 | Accuracy: 39.53%


In [10]:


def evaluate(model, dataloader):
    model.eval()
    correct1 = 0
    correct5 = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            _, top5 = torch.topk(logits, k=5, dim=1)
            preds = torch.argmax(logits, dim=1)

            correct1 += (preds == y_batch).sum().item()
            correct5 += (top5 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total += y_batch.size(0)

    print(f"Top-1 Accuracy: {100*correct1/total:.2f}%")
    print(f"Top-5 Accuracy: {100*correct5/total:.2f}%")


# Load
model.load_state_dict(torch.load("symbol_classifier.pt"))
model.eval()
evaluate(model,dataloader)

Top-1 Accuracy: 42.82%
Top-5 Accuracy: 91.84%
